# VLS Benchmark — Data & Experiment Visualization

This notebook covers:
1. **Dataset overview** — class balance, split sizes, protein coverage
2. **Ligand chemistry** — molecular weight, SMILES length, fingerprint density
3. **Model performance** — ROC-AUC / PR-AUC across train / val / test splits
4. **Generalization analysis** — train→test gap, per-metric comparison
5. **PDBbind comparison** — our results vs literature baselines

In [ ]:
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.1)
FIGSIZE = (10, 5)
COLORS = {"random_forest": "#4C72B0", "gradient_boosting": "#DD8452", "svm": "#55A868"}
SPLIT_COLORS = {"train": "#4C72B0", "val": "#DD8452", "test": "#55A868"}

# ── paths ──────────────────────────────────────────────────────────────
ROOT = Path("../").resolve()
REGISTRY  = ROOT / "training_data_full/registry.csv"
MODELS_DIR = ROOT / "benchmarks/02_training/trained_models"
REPORT_CSV = ROOT / "benchmarks/03_analysis/report.csv"

print("ROOT:", ROOT)
print("Registry exists:", REGISTRY.exists())
print("Models dir exists:", MODELS_DIR.exists())

## 1  Dataset Overview

In [ ]:
print("Loading registry...")
reg = pd.read_csv(REGISTRY)
print(f"Total rows: {len(reg):,}")
reg.head(2)

In [ ]:
# ── split × threshold breakdown ────────────────────────────────────────
summary = (
    reg.groupby(["similarity_threshold", "split"])
    .agg(
        n_total=("smiles", "count"),
        n_active=("is_active", "sum"),
        n_proteins=("uniprot_id", "nunique"),
        n_unique_smiles=("smiles", "nunique"),
    )
    .reset_index()
)
summary["pct_active"] = (summary["n_active"] / summary["n_total"] * 100).round(2)
summary

In [ ]:
# Focus on 0p7 threshold (used for training)
df07 = summary[summary["similarity_threshold"] == "0p7"].copy()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Total samples per split
axes[0].bar(df07["split"], df07["n_total"] / 1e6,
            color=[SPLIT_COLORS.get(s, "grey") for s in df07["split"]])
axes[0].set_title("Total samples per split")
axes[0].set_ylabel("Millions")

# Active rate per split
axes[1].bar(df07["split"], df07["pct_active"],
            color=[SPLIT_COLORS.get(s, "grey") for s in df07["split"]])
axes[1].set_title("Active rate (%) per split")
axes[1].set_ylabel("%")
axes[1].axhline(5, ls="--", color="red", alpha=0.5, label="5% baseline")
axes[1].legend()

# Unique proteins per split
axes[2].bar(df07["split"], df07["n_proteins"],
            color=[SPLIT_COLORS.get(s, "grey") for s in df07["split"]])
axes[2].set_title("Unique proteins per split")
axes[2].set_ylabel("Count")

fig.suptitle("Dataset overview — 0p7 similarity threshold", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Active vs decoy stacked bar across ALL thresholds
fig, ax = plt.subplots(figsize=FIGSIZE)
pivot = summary.pivot_table(index=["similarity_threshold", "split"],
                             values=["n_active", "n_total"]).reset_index()
pivot["n_decoy"] = pivot["n_total"] - pivot["n_active"]
pivot["label"] = pivot["similarity_threshold"] + " / " + pivot["split"]

x = range(len(pivot))
ax.bar(x, pivot["n_decoy"] / 1e6, label="Decoy", color="#4C72B0", alpha=0.8)
ax.bar(x, pivot["n_active"] / 1e6, bottom=pivot["n_decoy"] / 1e6,
       label="Active", color="#DD8452", alpha=0.9)
ax.set_xticks(list(x))
ax.set_xticklabels(pivot["label"], rotation=45, ha="right", fontsize=9)
ax.set_ylabel("Millions")
ax.set_title("Active vs Decoy counts — all thresholds & splits")
ax.legend()
plt.tight_layout()
plt.show()

## 2  Ligand Chemistry

In [ ]:
# Sample for heavy computation — use full unique SMILES set if memory allows
SAMPLE_N = 50_000
smiles_sample = reg["smiles"].dropna().drop_duplicates().sample(SAMPLE_N, random_state=42)

from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem

mols, mw_list, logp_list, hbd_list, hba_list, tpsa_list, rot_list, fp_density = [], [], [], [], [], [], [], []

for smi in smiles_sample:
    mol = Chem.MolFromSmiles(smi)
    if mol is None:
        continue
    mw_list.append(Descriptors.MolWt(mol))
    logp_list.append(Descriptors.MolLogP(mol))
    hbd_list.append(Descriptors.NumHDonors(mol))
    hba_list.append(Descriptors.NumHAcceptors(mol))
    tpsa_list.append(Descriptors.TPSA(mol))
    rot_list.append(Descriptors.NumRotatableBonds(mol))
    fp = AllChem.GetMorganFingerprintAsBitVect(mol, 2, nBits=2048)
    fp_density.append(fp.GetNumOnBits() / 2048)

chem_df = pd.DataFrame({
    "MolWt": mw_list, "LogP": logp_list, "HBD": hbd_list,
    "HBA": hba_list, "TPSA": tpsa_list, "RotBonds": rot_list,
    "FP_density": fp_density
})
print(f"Valid molecules: {len(chem_df):,} / {SAMPLE_N:,}")
chem_df.describe().round(2)

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

props = [
    ("MolWt",      "Molecular Weight (Da)",  (0, 1000)),
    ("LogP",       "LogP",                   (-5, 10)),
    ("TPSA",       "TPSA (Å²)",              (0, 200)),
    ("HBD",        "H-Bond Donors",          (-0.5, 10)),
    ("RotBonds",   "Rotatable Bonds",        (-0.5, 20)),
    ("FP_density", "Morgan FP bit density",  (0, 0.3)),
]

for ax, (col, label, xlim) in zip(axes, props):
    data = chem_df[col].clip(*xlim)
    ax.hist(data, bins=50, color="#4C72B0", alpha=0.8, edgecolor="white", linewidth=0.3)
    ax.set_xlabel(label)
    ax.set_ylabel("Count")
    ax.set_xlim(xlim)
    med = data.median()
    ax.axvline(med, color="#DD8452", ls="--", lw=1.5, label=f"median={med:.1f}")
    ax.legend(fontsize=8)

fig.suptitle(f"Ligand physicochemical properties (n={len(chem_df):,} sampled)",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 3  Model Performance

In [ ]:
report = pd.read_csv(REPORT_CSV)
report

In [ ]:
# ROC-AUC across train / val / test — grouped bar chart
models = report["model"].tolist()
x = np.arange(len(models))
width = 0.25

fig, ax = plt.subplots(figsize=FIGSIZE)
bars_train = ax.bar(x - width, report["train_roc_auc"], width, label="Train",
                    color=SPLIT_COLORS["train"], alpha=0.85)
bars_val   = ax.bar(x,         report["val_roc_auc"],   width, label="Val",
                    color=SPLIT_COLORS["val"],   alpha=0.85)
bars_test  = ax.bar(x + width, report["test_roc_auc"],  width, label="Test",
                    color=SPLIT_COLORS["test"],  alpha=0.85)

# Reference line: random
ax.axhline(0.5, color="black", ls=":", lw=1.2, label="Random (0.5)")

# Annotate bars
for bars in [bars_train, bars_val, bars_test]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2, h + 0.005,
                f"{h:.3f}", ha="center", va="bottom", fontsize=7.5)

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in models])
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0, 1.05)
ax.set_title("ROC-AUC by model and split (0p7 similarity threshold)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Average Precision (PR-AUC) — same layout
fig, ax = plt.subplots(figsize=FIGSIZE)
ax.bar(x - width, report["train_avg_precision"], width, label="Train",
       color=SPLIT_COLORS["train"], alpha=0.85)
ax.bar(x,         report["val_avg_precision"],   width, label="Val",
       color=SPLIT_COLORS["val"],   alpha=0.85)
ax.bar(x + width, report["test_avg_precision"],  width, label="Test",
       color=SPLIT_COLORS["test"],  alpha=0.85)

# Baseline: random classifier AP ≈ prevalence (~6%)
prevalence = 0.061
ax.axhline(prevalence, color="black", ls=":", lw=1.2, label=f"Random AP (≈{prevalence:.2f})")

ax.set_xticks(x)
ax.set_xticklabels([m.replace("_", "\n") for m in models])
ax.set_ylabel("Average Precision (PR-AUC)")
ax.set_ylim(0, 0.35)
ax.set_title("Average Precision by model and split", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Radar / spider chart — per-model multi-metric on TEST set
from matplotlib.patches import FancyArrowPatch

metrics = ["test_roc_auc", "test_avg_precision", "test_f1_score",
           "test_recall", "test_precision", "test_accuracy"]
labels  = ["ROC-AUC", "Avg Precision", "F1", "Recall", "Precision", "Accuracy"]

# Normalise: roc_auc & accuracy centred on 0.5, rest on 0
def normalise(col, val):
    if col in ("test_roc_auc", "test_accuracy"):
        return max(0, (val - 0.5) / 0.5)  # 0 = random, 1 = perfect
    return val

N = len(metrics)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig, ax = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for _, row in report.iterrows():
    vals = [normalise(m, row[m]) for m in metrics] + [normalise(metrics[0], row[metrics[0]])]
    color = COLORS.get(row["model"], "grey")
    ax.plot(angles, vals, color=color, lw=2, label=row["model"])
    ax.fill(angles, vals, color=color, alpha=0.12)

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10)
ax.set_title("Test-set metrics (normalised)\nROC-AUC & Accuracy: 0=random, 1=perfect",
             fontsize=10, pad=20)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1))
plt.tight_layout()
plt.show()

## 4  Generalization Analysis

In [ ]:
# Melt to long form for easier plotting
long_rows = []
for _, row in report.iterrows():
    for split in ["train", "val", "test"]:
        long_rows.append({
            "model": row["model"],
            "split": split,
            "roc_auc":       row[f"{split}_roc_auc"],
            "avg_precision": row[f"{split}_avg_precision"],
            "f1":            row[f"{split}_f1_score"],
            "recall":        row[f"{split}_recall"],
        })
long_df = pd.DataFrame(long_rows)
long_df["split"] = pd.Categorical(long_df["split"], ["train", "val", "test"])

In [ ]:
# ROC-AUC learning curve (train→val→test) per model
fig, ax = plt.subplots(figsize=FIGSIZE)

for model, grp in long_df.groupby("model"):
    grp = grp.sort_values("split")
    color = COLORS.get(model, "grey")
    ax.plot(grp["split"], grp["roc_auc"], marker="o", lw=2.5, color=color, label=model)
    for _, r in grp.iterrows():
        ax.annotate(f"{r['roc_auc']:.3f}",
                    (r["split"], r["roc_auc"]),
                    textcoords="offset points", xytext=(0, 8),
                    ha="center", fontsize=8, color=color)

ax.axhline(0.5, color="black", ls=":", lw=1.2, label="Random")
ax.set_ylabel("ROC-AUC")
ax.set_ylim(0.2, 1.0)
ax.set_title("Generalization curve: Train → Val → Test", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Generalization gap: train_auc − test_auc
report["gen_gap"] = report["train_roc_auc"] - report["test_roc_auc"]

fig, ax = plt.subplots(figsize=(7, 4))
colors = [COLORS.get(m, "grey") for m in report["model"]]
bars = ax.barh(report["model"], report["gen_gap"], color=colors, alpha=0.85)
for bar, val in zip(bars, report["gen_gap"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)
ax.set_xlabel("Generalization gap (Train ROC-AUC − Test ROC-AUC)")
ax.set_title("Overfitting severity (0p7 similarity split)", fontweight="bold")
ax.set_xlim(0, 0.7)
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap: all metrics × all splits
metric_cols = [
    "train_roc_auc", "val_roc_auc", "test_roc_auc",
    "train_avg_precision", "val_avg_precision", "test_avg_precision",
    "train_f1_score", "val_f1_score", "test_f1_score",
    "train_recall", "val_recall", "test_recall",
]
hm_data = report.set_index("model")[metric_cols]

fig, ax = plt.subplots(figsize=(13, 3.5))
sns.heatmap(hm_data, annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={"label": "metric value"})
ax.set_title("All metrics across train / val / test splits", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right", fontsize=9)
plt.tight_layout()
plt.show()

## 5  Virtual-Screening Metrics: Enrichment Factor & BEDROC

- **EF(χ%)**: ratio of actives found in top χ% vs random. EF=1 → random; EF=1/χ → perfect.
  Fractions evaluated: 0.1%, 0.2%, 0.5%, 1%, 2%, 5%, 10%, 15%, 20%
- **BEDROC(α)**: exponentially weights early-ranked actives (Truchon & Bayly 2007).
  α=20 ≈ top 8% weighted; α=80 ≈ top 2%; α=160 ≈ top 1%.

**Experiments**:
- *1D-VS*: 1D ligand similarity split with decoys → full VS evaluation (EF/BEDROC valid)
- *2D-Gen*: protein-cluster × ligand-similarity split → protein generalization test (actives only)

In [ ]:
# Load pre-computed VS metrics
vs_data = {}
for model_name in ["random_forest", "gradient_boosting", "svm"]:
    p = MODELS_DIR / f"{model_name}_vs_metrics.json"
    if p.exists():
        with open(p) as f:
            vs_data[model_name] = json.load(f)
        print(f"{model_name}: loaded")
    else:
        print(f"{model_name}: not found — run evaluate_vs_metrics.py first")

# Build a flat DataFrame
vs_rows = []
for model, d in vs_data.items():
    row = {"model": model}
    row.update(d["metrics"])
    row["prevalence"] = d["prevalence"]
    vs_rows.append(row)
vs_df = pd.DataFrame(vs_rows)
vs_df

In [ ]:
# Enrichment Factor at multiple levels
ef_cols = [c for c in vs_df.columns if c.startswith("ef_")]

def ef_col_to_label(col):
    s = col.replace("ef_", "").replace("pct", "")
    v = float(s.replace("p", ".")) if "p" in s else float(s)
    return f"{v}%"

ef_labels = [ef_col_to_label(c) for c in ef_cols]

fig, ax = plt.subplots(figsize=(13, 5))
x = np.arange(len(ef_cols))
width = 0.25
offsets = np.linspace(-width, width, len(vs_df))

for i, (_, row) in enumerate(vs_df.iterrows()):
    color = COLORS.get(row["model"], "grey")
    ax.bar(x + offsets[i], [row[c] for c in ef_cols],
           width * 0.9, label=row["model"], color=color, alpha=0.85)

ax.axhline(1.0, color="black", ls=":", lw=1.5, label="Random (EF=1)")
ax.set_xticks(x)
ax.set_xticklabels(ef_labels)
ax.set_ylabel("Enrichment Factor")
ax.set_xlabel("Top-χ fraction screened")
ax.set_title("Enrichment Factor at different screening fractions (test set)", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# EF enrichment curve — log x-axis shows early recognition clearly
fig, ax = plt.subplots(figsize=(10, 5))

def ef_col_to_frac_pct(col):
    s = col.replace("ef_", "").replace("pct", "")
    return float(s.replace("p", ".")) if "p" in s else float(s)

fracs_pct = [ef_col_to_frac_pct(c) for c in ef_cols]

for _, row in vs_df.iterrows():
    color = COLORS.get(row["model"], "grey")
    ef_vals = [row[c] for c in ef_cols]
    ax.plot(fracs_pct, ef_vals, marker="o", lw=2, color=color, label=row["model"])
    for frac, val in zip(fracs_pct, ef_vals):
        ax.annotate(f"{val:.2f}", (frac, val),
                    textcoords="offset points", xytext=(0, 6),
                    ha="center", fontsize=7, color=color)

ax.axhline(1.0, color="black", ls=":", lw=1.5, label="Random")
ax.set_xscale("log")
ax.set_xlabel("Top-χ% screened (log scale)")
ax.set_ylabel("Enrichment Factor")
ax.set_title("Enrichment curve — actives recovered vs random", fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
bedroc_cols = [c for c in vs_df.columns if c.startswith("bedroc_")]
bedroc_labels = {
    "bedroc_a20":  "BEDROC a=20 (top ~8%)",
    "bedroc_a80":  "BEDROC a=80 (top ~2%)",
    "bedroc_a160": "BEDROC a=160 (top ~1%)",
}

fig, axes = plt.subplots(1, len(bedroc_cols), figsize=(5 * len(bedroc_cols), 4))
if len(bedroc_cols) == 1: axes = [axes]

for ax, col in zip(axes, bedroc_cols):
    bar_colors = [COLORS.get(m, "grey") for m in vs_df["model"]]
    bars = ax.bar(vs_df["model"], vs_df[col], color=bar_colors, alpha=0.85)
    for bar, val in zip(bars, vs_df[col]):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
                f"{val:.4f}", ha="center", va="bottom", fontsize=9)
    prevalence = vs_df["prevalence"].mean()
    ax.axhline(prevalence, color="black", ls=":", lw=1.5,
               label=f"~Random ({prevalence:.3f})")
    ax.set_title(bedroc_labels.get(col, col), fontweight="bold")
    ax.set_ylabel("BEDROC")
    ax.set_ylim(0, max(float(vs_df[col].max()) * 1.3, 0.15))
    ax.set_xticklabels([m.replace("_", " ") for m in vs_df["model"]])
    ax.legend(fontsize=8)

fig.suptitle("BEDROC — early recognition of actives in ranked list", fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
display_cols = ["model"] + ef_cols + bedroc_cols
col_labels = ["Model"] + ef_labels + [bedroc_labels.get(c, c) for c in bedroc_cols]
summary_tbl = vs_df[display_cols].copy()
summary_tbl.columns = col_labels
summary_tbl = summary_tbl.set_index("Model")

fig, ax = plt.subplots(figsize=(16, 2.5))
sns.heatmap(summary_tbl.astype(float), annot=True, fmt=".3f",
            cmap="YlOrRd", linewidths=0.5, ax=ax,
            cbar_kws={"label": "metric value"})
ax.set_title("VS metrics heatmap — higher is better", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=35, ha="right")
plt.tight_layout()
plt.show()

print("\nRaw values:")
print(summary_tbl.round(4).to_string())

## 5  Training Efficiency

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Training time
axes[0].barh(report["model"], report["training_time_s"] / 60,
             color=[COLORS.get(m, "grey") for m in report["model"]], alpha=0.85)
axes[0].set_xlabel("Training time (minutes)")
axes[0].set_title("Training time")
for i, (_, row) in enumerate(report.iterrows()):
    axes[0].text(row["training_time_s"] / 60 + 0.3, i, f"{row['training_time_s']/60:.1f} min",
                 va="center", fontsize=9)

# ROC-AUC per minute of training
report["auc_per_min"] = report["test_roc_auc"] / (report["training_time_s"] / 60)
axes[1].barh(report["model"], report["auc_per_min"],
             color=[COLORS.get(m, "grey") for m in report["model"]], alpha=0.85)
axes[1].set_xlabel("Test ROC-AUC per minute of training")
axes[1].set_title("Efficiency (AUC / training-min)")

plt.suptitle("Training efficiency comparison", fontweight="bold")
plt.tight_layout()
plt.show()

## 6  PDBbind Comparison

In [ ]:
lit = pd.DataFrame([
    {"method": "RF + ECFP4 (lit.)",    "auc_lo": 0.72, "auc_hi": 0.78},
    {"method": "GBM + ECFP (lit.)",    "auc_lo": 0.74, "auc_hi": 0.80},
    {"method": "SVM + ECFP (lit.)",    "auc_lo": 0.70, "auc_hi": 0.76},
    {"method": "DeepDTA CNN (lit.)",    "auc_lo": 0.74, "auc_hi": 0.78},
])

fig, ax = plt.subplots(figsize=(10, 5))

# Literature ranges (error bars)
for i, row in lit.iterrows():
    mid = (row["auc_lo"] + row["auc_hi"]) / 2
    err = (row["auc_hi"] - row["auc_lo"]) / 2
    ax.errorbar(mid, row["method"], xerr=err, fmt="s",
                color="#7f7f7f", markersize=8, lw=2, capsize=5)

# Our results
model_labels = {"random_forest": "RF (ours, 0p7)",
                "gradient_boosting": "GBM (ours, 0p7)",
                "svm": "SVM (ours, 0p7)"}
for _, row in report.iterrows():
    color = COLORS.get(row["model"], "grey")
    label = model_labels.get(row["model"], row["model"])
    ax.scatter(row["test_roc_auc"], label, marker="D",
               color=color, s=80, zorder=5, label=label)
    ax.annotate(f"{row['test_roc_auc']:.3f}",
                (row["test_roc_auc"], label),
                xytext=(-8, -12), textcoords="offset points",
                fontsize=8, color=color)

ax.axvline(0.5, color="black", ls=":", lw=1, label="Random")
ax.set_xlabel("Test ROC-AUC")
ax.set_xlim(0.25, 0.95)
ax.set_title("Our results vs PDBbind literature baselines\n"
             "(gap expected: literature uses random/loose splits; we use 0p7 similarity split)",
             fontweight="bold")
ax.legend(loc="lower right", fontsize=8)
ax.grid(axis="x", alpha=0.4)
plt.tight_layout()
plt.show()

## 7  Feature Cache Stats

In [ ]:
import h5py

cache_path = ROOT / "training_data_full/feature_cache/morgan_r2_b2048.h5"
with h5py.File(cache_path, "r") as f:
    n_cached = len(f["key_hashes"])
    feat_shape = f["features"].shape
    config = json.loads(f.attrs["config"])
    packed = f.attrs["packed_bits"]

size_mb = cache_path.stat().st_size / 1e6
naive_mb = n_cached * 2048 * 4 / 1e6  # float32

print(f"Cached entries : {n_cached:,}")
print(f"Feature shape  : {feat_shape}  (packed bits)")
print(f"Config         : {config}")
print(f"Cache file     : {size_mb:.1f} MB")
print(f"Naive float32  : {naive_mb:.0f} MB")
print(f"Compression    : {naive_mb/size_mb:.0f}×")

## 9  Experiment Comparison

Unified view of all experiments with both ROC-AUC and VS-specific metrics.
New training runs will appear here automatically once their summaries are saved.

In [ ]:
# Load all training summaries + VS metrics — auto-picks up new experiments
exp_rows = []
for p in sorted(MODELS_DIR.glob("*_training_summary.json")):
    model_name = p.stem.replace("_training_summary", "")
    with open(p) as f:
        s = json.load(f)
    row = {
        "model":     model_name,
        "split":     s.get("split_strategy", "1d_ligand_similarity"),
        "train_auc": s["training_history"]["train_metrics"]["roc_auc"],
        "val_auc":   s["training_history"]["val_metrics"]["roc_auc"],
        "test_auc":  s["training_history"]["test_metrics"]["roc_auc"],
        "gen_gap":   s["training_history"]["train_metrics"]["roc_auc"] -
                     s["training_history"]["test_metrics"]["roc_auc"],
    }
    vs_p = MODELS_DIR / f"{model_name}_vs_metrics.json"
    if vs_p.exists():
        with open(vs_p) as f:
            vs = json.load(f)
        for k in ["ef_0.1pct", "ef_1pct", "ef_5pct",
                  "bedroc_a20", "bedroc_a80", "bedroc_a160"]:
            row[k] = vs["metrics"].get(k, float("nan"))
    exp_rows.append(row)

exp_df = pd.DataFrame(exp_rows)
display_cols = [c for c in ["model", "split", "train_auc", "val_auc", "test_auc",
                             "gen_gap", "ef_1pct", "ef_5pct",
                             "bedroc_a20", "bedroc_a80", "bedroc_a160"]
                if c in exp_df.columns]
exp_df[display_cols].round(4)


In [ ]:
# ROC-AUC + EF@1% + BEDROC side by side for all experiments
metrics_to_plot = [
    ("test_auc",    "Test ROC-AUC"),
    ("ef_1pct",     "EF @ 1%"),
    ("bedroc_a20",  "BEDROC a=20"),
    ("bedroc_a160", "BEDROC a=160"),
]
valid = [(col, lbl) for col, lbl in metrics_to_plot if col in exp_df.columns]
fig, axes = plt.subplots(1, len(valid), figsize=(4 * len(valid), 4))
if len(valid) == 1:
    axes = [axes]

for ax, (col, label) in zip(axes, valid):
    bar_colors = [COLORS.get(m, "grey") for m in exp_df["model"]]
    ax.bar(range(len(exp_df)), exp_df[col].fillna(0), color=bar_colors, alpha=0.85)
    ax.set_xticks(range(len(exp_df)))
    ax.set_xticklabels(exp_df["model"].str.replace("_", " "),
                       fontsize=7, rotation=30, ha="right")
    ax.set_title(label, fontweight="bold")
    if col == "test_auc":
        ax.axhline(0.5, color="black", ls=":", lw=1, label="Random")
        ax.legend(fontsize=7)
    elif col.startswith("ef_"):
        ax.axhline(1.0, color="black", ls=":", lw=1, label="Random")
        ax.legend(fontsize=7)

fig.suptitle("All experiments — key VS metrics", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Scatter: test ROC-AUC vs EF@1% and gen-gap vs BEDROC a=20
plot_df = exp_df.dropna(subset=["ef_1pct", "bedroc_a20"])

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for _, row in plot_df.iterrows():
    color = COLORS.get(row["model"], "grey")
    lbl = row["model"].replace("_", " ")
    axes[0].scatter(row["test_auc"], row["ef_1pct"], s=130, color=color, zorder=5)
    axes[0].annotate(lbl, (row["test_auc"], row["ef_1pct"]),
                     textcoords="offset points", xytext=(5, 4), fontsize=8, color=color)
    axes[1].scatter(row["gen_gap"], row["bedroc_a20"], s=130, color=color, zorder=5)
    axes[1].annotate(lbl, (row["gen_gap"], row["bedroc_a20"]),
                     textcoords="offset points", xytext=(5, 4), fontsize=8, color=color)

axes[0].axhline(1.0, color="black", ls=":", lw=1)
axes[0].set_xlabel("Test ROC-AUC")
axes[0].set_ylabel("EF @ 1%")
axes[0].set_title("Discrimination vs Early Enrichment", fontweight="bold")

axes[1].set_xlabel("Generalization gap (train - test ROC-AUC)")
axes[1].set_ylabel("BEDROC a=20")
axes[1].set_title("Overfitting vs Early-Recognition Quality", fontweight="bold")

fig.suptitle("Key trade-offs across all experiments", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
# Full heatmap of all VS + ROC metrics across experiments
hm_cols = [c for c in ["train_auc", "val_auc", "test_auc", "gen_gap",
                        "ef_1pct", "ef_5pct", "bedroc_a20", "bedroc_a80", "bedroc_a160"]
           if c in exp_df.columns]
hm = exp_df.set_index("model")[hm_cols].fillna(0)

fig, ax = plt.subplots(figsize=(14, 2.5))
sns.heatmap(hm.astype(float), annot=True, fmt=".3f", cmap="RdYlGn",
            vmin=0, vmax=1, linewidths=0.5, ax=ax)
ax.set_title("All experiments — full metric heatmap", fontweight="bold")
ax.set_xticklabels(ax.get_xticklabels(), rotation=30, ha="right")
plt.tight_layout()
plt.show()
